In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/Nanchang_WC_Samples_15000.csv"
)

df.head()

,system:index,B11,B12,B2,B3,B4,B8,Class,MNDWI,NDBI,NDVI,lat,lon,.geo
0,0,2141.0,1673.5,513.0,594.5,652.5,1671.0,1,-0.565345,0.123295,0.438347,28.432891,116.472011,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,1,1918.0,1791.5,965.5,1190.0,1314.0,1800.0,1,-0.234234,0.031737,0.156069,28.426873,115.964284,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,2,1857.0,1297.0,683.0,923.0,937.5,2546.0,1,-0.335971,-0.156484,0.461748,28.548684,115.872566,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,3,2108.0,1886.0,792.0,1040.0,1257.0,1819.5,1,-0.339263,0.073456,0.182838,28.643996,115.956558,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,4,2500.5,2389.0,1408.0,1669.0,1884.0,2324.0,1,-0.199424,0.036584,0.104563,28.686127,115.855048,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [2]:
print(df.shape)

print(df.columns.tolist())

(15000, 14)
['system:index', 'B11', 'B12', 'B2', 'B3', 'B4', 'B8', 'Class', 'MNDWI', 'NDBI', 'NDVI', 'lat', 'lon', '.geo']


In [3]:
df['Class'].value_counts()

Class
1    5000
2    5000
3    5000
Name: count, dtype: int64

In [4]:
df.isnull().sum()

system:index    0
B11             0
B12             0
B2              0
B3              0
B4              0
B8              0
Class           0
MNDWI           0
NDBI            0
NDVI            0
lat             0
lon             0
.geo            0
dtype: int64

In [5]:
print(df.shape)
print(df.columns.tolist())
df['Class'].value_counts()

(15000, 14)
['system:index', 'B11', 'B12', 'B2', 'B3', 'B4', 'B8', 'Class', 'MNDWI', 'NDBI', 'NDVI', 'lat', 'lon', '.geo']


Class
1    5000
2    5000
3    5000
Name: count, dtype: int64

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv(
    "../data/Nanchang_WC_Samples_15000.csv"
)

features = [
    'B2','B3','B4',
    'B8','B11','B12',
    'NDVI','NDBI','MNDWI'
]

X = df[features]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "OA:",
    accuracy_score(y_test, pred)
)


OA: 0.898


In [10]:
import numpy as np

prob = rf.predict_proba(X_test)

prob[:5]

array([[0.64, 0.36, 0.  ],
       [0.18, 0.82, 0.  ],
       [0.53, 0.47, 0.  ],
       [0.  , 0.03, 0.97],
       [0.  , 0.  , 1.  ]])

In [11]:
sorted_prob = np.sort(
    prob,
    axis=1
)

margin = (
    sorted_prob[:, -1]
    -
    sorted_prob[:, -2]
)

margin[:10]

array([0.28, 0.64, 0.06, 0.94, 1.  , 0.96, 0.32, 0.85, 0.94, 0.98])

In [12]:
hard_idx = np.argsort(margin)

hard_idx[:10]


array([2101, 2580,  831, 2715,  708,  319,  579, 2105, 2125, 2204])

In [13]:
hard_samples = (
    X_test.iloc[hard_idx[:100]]
)

hard_samples.head()

,B2,B3,B4,B8,B11,B12,NDVI,NDBI,MNDWI
9342,894.000000,1365.5,1415.000000,2320.0,1872.0,1282.0,0.242303,-0.106870,-0.156448
10890,504.500000,794.5,636.000000,2422.5,1573.5,1100.5,0.584110,-0.212462,-0.328970
12612,670.000000,999.0,946.000000,1923.0,1572.0,1063.0,0.340537,-0.100429,-0.222870
9999,785.666667,1147.0,1125.333333,2568.0,2403.0,1787.0,0.390614,-0.033193,-0.353803
11843,1587.000000,2014.0,2425.000000,2826.0,3868.0,3461.5,0.076366,0.155662,-0.315199


In [14]:
hard_samples = df.loc[
    X_test.iloc[hard_idx[:100]].index,
    ['lon','lat','Class']
]

hard_samples.head()


,lon,lat,Class
9342,115.980633,29.024342,2
10890,116.428982,28.670586,3
12612,115.906522,28.785840,3
9999,115.987819,28.409356,2
11843,116.098402,28.513560,3


In [16]:
from pathlib import Path

out = Path("../outputs")
out.mkdir(parents=True, exist_ok=True)

hard_samples.to_csv(
    out / "top100_uncertain.csv",
    index=False
)
print("saved:", out.resolve() / "top100_uncertain.csv")

saved: /Users/alex/Projects/gee-project/outputs/top100_uncertain.csv


In [17]:
prob[:5]

margin[:20]

hard_samples.head()

,lon,lat,Class
9342,115.980633,29.024342,2
10890,116.428982,28.670586,3
12612,115.906522,28.785840,3
9999,115.987819,28.409356,2
11843,116.098402,28.513560,3


In [18]:
print("Probability:")
print(prob[:5])

print("\nMargin:")
print(margin[:20])

print("\nHard Samples:")
hard_samples.head()

Probability:
[[0.64 0.36 0.  ]
 [0.18 0.82 0.  ]
 [0.53 0.47 0.  ]
 [0.   0.03 0.97]
 [0.   0.   1.  ]]

Margin:
[0.28 0.64 0.06 0.94 1.   0.96 0.32 0.85 0.94 0.98 0.73 0.96 0.91 0.64
 0.98 0.88 1.   1.   0.89 0.68]

Hard Samples:


,lon,lat,Class
9342,115.980633,29.024342,2
10890,116.428982,28.670586,3
12612,115.906522,28.785840,3
9999,115.987819,28.409356,2
11843,116.098402,28.513560,3


In [19]:
hard_samples = df.loc[
    X_test.iloc[hard_idx[:100]].index,
    ['lon', 'lat', 'Class']
].copy()

hard_samples['margin'] = margin[
    hard_idx[:100]
]

hard_samples.to_csv(
    "../data/top100_uncertain_samples.csv",
    index=False
)

hard_samples.head()

,lon,lat,Class,margin
9342,115.980633,29.024342,2,0.0
10890,116.428982,28.670586,3,0.0
12612,115.906522,28.785840,3,0.0
9999,115.987819,28.409356,2,0.0
11843,116.098402,28.513560,3,0.0


In [20]:
len(hard_samples)

100

In [21]:
hard_samples["margin"].describe()

count    100.000000
mean       0.037000
std        0.021858
min        0.000000
25%        0.020000
50%        0.040000
75%        0.060000
max        0.070000
Name: margin, dtype: float64

In [22]:
hard_samples.to_string()

'              lon        lat  Class  margin\n9342   115.980633  29.024342      2    0.00\n10890  116.428982  28.670586      3    0.00\n12612  115.906522  28.785840      3    0.00\n9999   115.987819  28.409356      2    0.00\n11843  116.098402  28.513560      3    0.00\n5526   115.892957  28.462805      2    0.00\n11210  116.305104  28.714154      3    0.00\n3087   115.541357  28.740475      1    0.01\n11499  115.812468  28.359319      3    0.01\n4196   116.308877  28.272902      1    0.01\n9244   115.945868  28.572400      2    0.01\n4125   116.169369  28.233465      1    0.01\n14568  116.218417  29.063958      3    0.01\n7333   115.899964  28.807758      2    0.01\n11773  116.035520  28.687294      3    0.01\n11824  115.980813  28.842613      3    0.01\n5797   116.236293  28.607254      2    0.01\n9155   115.793244  28.405493      2    0.01\n6553   116.310854  29.053088      2    0.01\n10443  116.023033  29.071683      3    0.02\n6182   116.133796  28.990835      2    0.02\n14253  11

In [23]:
human_labels = {
  1:2, 2:2, 3:2, 4:2, 5:2,
  6:2, 7:3, 8:1, 9:2, 10:2,
  11:2, 12:1, 13:3, 14:2, 15:2,
  16:3, 17:3, 18:2, 19:3, 20:3,

  21:2,22:2,23:3,24:1,25:2,
  26:2,27:3,28:2,29:3,30:2,
  31:2,32:1,33:3,34:3,35:2,
  36:2,37:1,38:2,39:3,40:2,

  41:2,42:2,43:3,44:2,45:2,
  46:2,47:2,48:2,49:2,50:2,
  51:3,52:1,53:2,54:2,55:3,
  56:2,57:2,58:3,59:3,60:2,

  61:2,62:3,63:1,64:2,65:3,
  66:3,67:3,68:3,69:3,70:1,
  71:2,72:2,73:2,74:3,75:3,
  76:2,77:1,78:3,79:3,80:3,

  81:2,82:2,83:1,84:2,85:3,
  86:2,87:3,88:3,89:1,90:1,
  91:2,92:3,93:3,94:2,95:2,
  96:1,97:2,98:3,99:2,100:2
}


In [24]:
hard_samples = hard_samples.reset_index(drop=True)

hard_samples["PointID"] = range(
    1,
    len(hard_samples)+1
)

hard_samples["WC_Class"] = (
    hard_samples["Class"]
)

hard_samples["Human_Class"] = (
    hard_samples["PointID"]
    .map(human_labels)
)

top100_corrected = hard_samples[
    [
        "PointID",
        "lon",
        "lat",
        "WC_Class",
        "Human_Class",
        "margin"
    ]
]

top100_corrected.head()

,PointID,lon,lat,WC_Class,Human_Class,margin
0,1,115.980633,29.024342,2,2,0.0
1,2,116.428982,28.670586,3,2,0.0
2,3,115.906522,28.785840,3,2,0.0
3,4,115.987819,28.409356,2,2,0.0
4,5,116.098402,28.513560,3,2,0.0


In [25]:
top100_corrected.to_csv(
    "../data/top100_corrected.csv",
    index=False
)

print(
    "Saved top100_corrected.csv"
)

Saved top100_corrected.csv


In [26]:
top100_corrected.shape


(100, 6)

In [27]:
top100_corrected.head()

,PointID,lon,lat,WC_Class,Human_Class,margin
0,1,115.980633,29.024342,2,2,0.0
1,2,116.428982,28.670586,3,2,0.0
2,3,115.906522,28.785840,3,2,0.0
3,4,115.987819,28.409356,2,2,0.0
4,5,116.098402,28.513560,3,2,0.0
